# 🧳 Exercise: Intelligent Travel Assistant

## Learning Objectives
In this exercise, you will practice:
1. **Custom tools** - Writing two `@tool` functions over real APIs (Tavily web search, OpenWeatherMap)
2. **Tool descriptions** - Docstrings good enough that the model calls the right tool with the right input
3. **A system prompt for a use case** - Making the agent always return attractions *and* weather
4. **A ReAct agent** - Building and running it with `create_agent`

## 🎯 The Assignment

Build an **Intelligent Travel Assistant** that, given a destination, can:
1. **Search for local attractions** in the destination
2. **Fetch current weather** for the destination
3. **Provide travel-ready information** combining both data sources

The full brief is `intelligent_travel_assistant_brief.pdf` in this folder. It names LangChain 0.x's `create_tool_calling_agent` + `AgentExecutor`; this course uses LangChain 1.x, where `create_agent` replaces both (see `03_react_with_create_agent.ipynb`). The skills practised are the same.

## 🏗️ Architecture

```
User Query: "Tell me about Kolkata"
              │
              ▼
┌─────────────────────────────────────────┐
│         Travel Assistant Agent          │
├─────────────────────────────────────────┤
│  System prompt: travel assistant that   │
│  returns attractions + current weather  │
└─────────────────────────────────────────┘
              │
    ┌─────────┴─────────┐
    ▼                   ▼
┌────────────┐    ┌─────────────┐
│ Web Search │    │   Weather   │
│    Tool    │    │    Tool     │
└────────────┘    └─────────────┘
    │                   │
    ▼                   ▼
┌────────────┐    ┌─────────────┐
│ Tavily API │    │ OpenWeather │
└────────────┘    └─────────────┘
```

## How to work through it
Each task is a cell marked **`TODO`** followed by a check cell. The notebook runs top to bottom as-is — unfinished tasks report `⏳ not done yet` instead of failing — so you can work one task at a time and re-run. A worked answer is in `solutions/03_react_agents/01_intelligent_travel_assistant_solution.ipynb`; try the tasks first.

## Prerequisites
- ReAct agents with `create_agent` — `03_react_with_create_agent.ipynb`
- Dependencies installed with `uv sync` at the repository root
- `OPENAI_API_KEY`, `TAVILY_API_KEY` and `WEATHER_API_KEY` (free key from [OpenWeatherMap](https://openweathermap.org/api)) in the project-root `.env`

---
## 📦 Step 1: Environment Setup

Load the keys (the cell only reports whether each is present), create the LLM, and set up the search plumbing your web-search tool will use.

In [1]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and check they are present
# ============================================================================
import os
import warnings

from dotenv import find_dotenv, load_dotenv

warnings.filterwarnings("ignore")

# Reads OPENAI_API_KEY, TAVILY_API_KEY and WEATHER_API_KEY from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

for key in ["OPENAI_API_KEY", "TAVILY_API_KEY", "WEATHER_API_KEY"]:
    print(f"{'✅' if os.getenv(key) else '❌ missing'} {key}")

✅ OPENAI_API_KEY
✅ TAVILY_API_KEY
✅ WEATHER_API_KEY


In [2]:
# ============================================================================
# LLM INITIALIZATION: The travel assistant's reasoning engine
# ============================================================================
from aied_helpers import get_openai_llm

llm = get_openai_llm(model="gpt-4o-mini")

print(f"✅ LLM initialized: {llm.model_name}")

✅ LLM initialized: gpt-4o-mini


In [3]:
# ============================================================================
# SEARCH PLUMBING: a Tavily search tool and a web-page-to-text converter
# ============================================================================
import requests
from langchain_tavily import TavilySearch
from markitdown import MarkItDown

# Tavily finds pages; basic depth is faster and enough for attraction queries
tavily_search = TavilySearch(max_results=3, search_depth="basic")

# Certain websites won't let you crawl them unless you send a browser-like user agent
session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/112.0.0.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
})
# MarkItDown converts a web page into readable text
md_converter = MarkItDown(requests_session=session)

print("✅ Tavily search and MarkItDown ready")

✅ Tavily search and MarkItDown ready


---
## 🌤️ Part 1: The Weather Tool (Task 1)

Write `get_weather(location)`. It should call OpenWeatherMap's current-weather endpoint,
`https://api.openweathermap.org/data/2.5/weather?q=<location>&appid=<WEATHER_API_KEY>&units=metric`, and return a short
readable summary (temperature, condition, humidity, wind). If the location is not found or the
request fails, return a message saying so rather than raising.

### Key Concepts:
- The `@tool` decorator converts a Python function into a LangChain tool
- **Docstrings are critical** - the LLM uses them to decide when to call each tool
- Tools should handle errors gracefully and return meaningful messages

In [4]:
# ============================================================================
# TODO (Task 1): Implement the weather tool
# ============================================================================
from langchain_core.tools import tool

WEATHER_API_KEY = os.getenv("WEATHER_API_KEY")


@tool
def get_weather(location: str) -> str:
    """TODO: describe what this tool does and when the model should use it."""
    # TODO: request https://api.openweathermap.org/data/2.5/weather with q=location,
    #       appid=WEATHER_API_KEY and units=metric, then format data["main"],
    #       data["weather"][0] and data["wind"] into a short summary.
    return "TODO: get_weather is not implemented yet"

In [5]:
# ============================================================================
# CHECK (Task 1): Call the tool directly
# ============================================================================
weather = get_weather.invoke("Kolkata")
if weather.startswith("TODO"):
    print("⏳ Task 1 not done yet — implement get_weather above, then re-run this cell.")
else:
    print(f"✅ Task 1 output:\n{weather}")

⏳ Task 1 not done yet — implement get_weather above, then re-run this cell.


---
## 🔍 Part 2: The Web-Search Tool (Task 2)

Write `search_web_extract_info(query)`. Use `tavily_search.invoke({"query": query})` to find
pages (the URLs are in `response["results"]`), convert each page to text with
`md_converter.convert(url).text_content`, and return a list of strings. Keep each page to about
1,500 characters so the agent's context stays small, and skip pages that fail or time out
(tip: fetch with `session.get(url, timeout=10)` and convert with `md_converter.convert_response(...)`,
because `md_converter.convert(url)` has no timeout).

In [6]:
# ============================================================================
# TODO (Task 2): Implement the web-search tool
# ============================================================================
@tool
def search_web_extract_info(query: str) -> list:
    """TODO: describe what this tool does and when the model should use it."""
    # TODO: search with tavily_search, then extract and truncate the text of each result URL.
    return ["TODO: search_web_extract_info is not implemented yet"]

In [7]:
# ============================================================================
# CHECK (Task 2): Call the tool directly
# ============================================================================
pages = search_web_extract_info.invoke("top attractions in Kolkata")
if pages and str(pages[0]).startswith("TODO"):
    print("⏳ Task 2 not done yet — implement search_web_extract_info above, then re-run this cell.")
else:
    print(f"✅ Task 2 returned {len(pages)} page(s); first one starts:\n{str(pages[0])[:300]}")

⏳ Task 2 not done yet — implement search_web_extract_info above, then re-run this cell.


---
## 🤖 Part 3: The Agent (Tasks 3 and 4)

**Task 3:** write `SYSTEM_PROMPT`. For any destination the user mentions, the agent should search
for attractions, get the current weather, and answer with the attractions as bullet points plus a
weather summary.

**Task 4:** build `travel_agent` with `create_agent(llm, tools=[...], system_prompt=SYSTEM_PROMPT)`.
Add `ModelCallLimitMiddleware(run_limit=6)` so a confused run cannot loop forever.

In [8]:
# ============================================================================
# TODO (Tasks 3 and 4): System prompt and agent
# ============================================================================
from langchain.agents import create_agent
from langchain.agents.middleware import ModelCallLimitMiddleware

SYSTEM_PROMPT = "TODO: write the travel assistant's system prompt"

# TODO: travel_agent = create_agent(...)
travel_agent = None

---
## 🧪 Part 4: Test the Travel Assistant

In [9]:
# ============================================================================
# CHECK (Tasks 3 and 4): Ask about a destination
# ============================================================================
from IPython.display import Markdown, display

if travel_agent is None or SYSTEM_PROMPT.startswith("TODO"):
    print("⏳ Tasks 3–4 not done yet — write SYSTEM_PROMPT and build travel_agent, then re-run.")
else:
    result = travel_agent.invoke({"messages": [("user", "Kolkata")]})
    used = [tc["name"] for m in result["messages"] for tc in getattr(m, "tool_calls", [])]
    print(f"🔧 Tools called: {used}")
    display(Markdown(result["messages"][-1].content))

⏳ Tasks 3–4 not done yet — write SYSTEM_PROMPT and build travel_agent, then re-run.


---
## 📝 Summary

When every check passes, you have built an agent that:
- ✅ Uses web search to find local attractions
- ✅ Fetches real-time weather data
- ✅ Combines both into helpful travel information
- ✅ Decides which tools to call, reads their results, and answers — the ReAct loop

### Key Concepts Practiced
| Concept | Implementation |
|---------|---------------|
| Custom Tools | `@tool` decorator over Tavily and OpenWeatherMap |
| Tool descriptions | Docstrings that tell the model when to use each tool |
| System Prompt | Travel-specific instructions: attractions *and* weather |
| ReAct agent | `create_agent` with a model-call limit |

### Bonus Exercises
1. **Add Memory**: give the agent a checkpointer and a `thread_id` so it remembers the user's preferences
2. **More Tools**: add tools for flight prices, hotel recommendations, or restaurant search
3. **Multi-City**: extend the agent to compare multiple destinations
4. **Streaming**: use `stream` instead of `invoke` for real-time responses

### Next Steps
- **`solutions/03_react_agents/01_intelligent_travel_assistant_solution.ipynb`**: a worked answer, with a walk-through of how the agent reasons